In [1]:
#| default_exp datasources.aisgeo
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

# Insert in Path Project Directory
sys.path.insert(0, str(Path().cwd().parent))
# os.chdir(Path.cwd().parent / 'src')

# GEOAISWEB

> Este módulo concentra as constantes, funções de carga, processamento, mesclagem e salvamento de dados aeronáuticos provenientes da API do GeoAisWeb

In [1]:
#| export
import json
import os
from pathlib import Path
from typing import List
from urllib.request import urlopen

import pandas as pd
from rfdatahub.constants import VOR_ILS_DME

## CONSTANTES


Dados para acesso à API GEOAISWEB

In [2]:
#| export
LINK_VOR = "https://geoaisweb.decea.mil.br/geoserver/ICA/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=ICA:vor&outputFormat=application%2Fjson"
LINK_DME = "https://geoaisweb.decea.mil.br/geoserver/ICA/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=ICA:dme&outputFormat=application%2Fjson"
LINK_NDB = "https://geoaisweb.decea.mil.br/geoserver/ICA/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=ICA:ndb&outputFormat=application%2Fjson"
COLS_VOR = (
    "properties.frequency",
    "properties.frequnits",
    "properties.latitude",
    "properties.longitude",
    "properties.tipo",
    "properties.txtname",
    "properties.txtrmk",
)
COLS_NDB = (
    "properties.valfreq",
    "properties.uomfreq",
    "properties.geolat",
    "properties.geolong",
    "properties.tipo",
    "properties.txtname",
    "properties.txtrmk",
)

COLS_DME = (
    "properties.valchannel",
    "properties.codechanne",
    "properties.geolat",
    "properties.geolong",
    "properties.tipo",
    "properties.txtname",
    "Channel",
)

UNIQUE_COLS = ["Frequência", "Latitude", "Longitude"]

In [3]:
def convert_frequency(
    freq: float,  # Frequência Central da emissão
    unit: str,    # Unidade da Frequência: [Hz, kHz, MHz, GHz]
) -> float:       # Frequência em MHz
    """Converte a frequência `freq` para MHz"""
    conversion = {
        'GHZ': lambda x: x * 1000,
        'MHZ': lambda x: x,
        'KHZ': lambda x: x / 1000,
        'HZ': lambda x: x / 1e6
    }
    unit = unit.upper()
    return conversion.get(unit, lambda _: -1.0)(freq)

In [4]:
def _process_frequency(
    df: pd.DataFrame,  # Dataframe com os dados
    cols: List[str],   # Subconjunto de Colunas relevantes
) -> pd.DataFrame:
    if cols == COLS_DME:
        df_channels = pd.read_csv(VOR_ILS_DME, dtype={'Channel': str, 'DMEground': float})
        df = df.dropna(subset=[cols[0]]).copy()
        
        # Vectorized channel creation
        df['Channel'] = df[cols[0]].astype("int").astype("string") + df[cols[1]].str.upper()
        
        
        # Merge instead of iterative lookup
        df = df.merge(
            df_channels[['Channel', 'DMEground']],
            on='Channel',
            how='left'
        ).rename(columns={'DMEground': 'Frequência'})
        
    else:
        # Vectorized frequency conversion
        df['Frequência'] = (
            df[[cols[0], cols[1]]]
            .apply(lambda x: convert_frequency(x.iloc[0], x.iloc[1]), axis=1)
            .astype("float")
        )
    
    return df

In [5]:
def _filter_df(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    # Use vectorized string operations
    df = df.assign(
        Entidade=(
            df[cols[4]].fillna('') + ' - ' +
            df[cols[5]].fillna('') + ' ' +
            df[cols[6]].fillna('')
        ).astype("string").str.strip(),
        Fonte=pd.Categorical(len(df) * ['AISGEO'], categories=['AISGEO'])
    )
    
    return (
        df[['Frequência', cols[2], cols[3], 'Entidade', 'Fonte']]
        .rename(columns={cols[2]: 'Latitude', cols[3]: 'Longitude'})
        .astype({'Frequência': float, 'Latitude': float, 'Longitude': float})
    )


In [6]:
def get_geodf(
    link: str,   # Link para a requisição
    cols: List[str],  # Colunas relevantes
) -> pd.DataFrame:
    with urlopen(link) as response:
        if response.status != 200 or not response.headers.get('content-type', '').startswith('application/json'):
            raise ValueError(f"Resposta a requisição não foi bem sucedida: {response.status=}")
        
        data = json.load(response)
    
    return (
        pd.json_normalize(data['features'])
        .filter(cols)
        .pipe(_process_frequency, cols=cols)
        .pipe(_filter_df, cols=cols)
    )

In [7]:
#| eval: false
get_geodf(LINK_VOR, COLS_VOR)

,Frequência,Latitude,Longitude,Entidade,Fonte
0,112.1,-25.583203,-54.503514,VOR - FOZ,AISGEO
1,116.7,-23.890979,-46.528199,VOR - REDE,AISGEO
2,114.3,-32.342167,-54.222000,VOR - MELO,AISGEO
3,116.5,-12.906418,-38.321364,VOR - SALVADOR,AISGEO
4,115.9,-22.344447,-41.769000,VOR - MACAÉ,AISGEO
...,...,...,...,...,...
75,117.7,4.693000,-61.028833,VOR - LA DIVINA PASTORA,AISGEO
76,115.9,-14.799000,-64.938333,"VOR - TRINIDAD-BL For ADDN INFO, check Bolivia...",AISGEO
77,115.6,-21.142778,-47.770278,VOR - RIBEIRAO PRETO,AISGEO
78,113.3,-31.718883,-52.327447,VOR - PELOTAS,AISGEO


In [8]:
#| eval: false
get_geodf(LINK_NDB, COLS_NDB)

,Frequência,Latitude,Longitude,Entidade,Fonte
0,0.1143,-32.340056,-54.223889,NDB - MELO,AISGEO
1,0.3000,-25.402616,-49.228917,NDB - BACACHERI NDB NOT AVBL BEYOND 50 NM,AISGEO
2,0.3400,-22.168824,-49.069677,NDB - AREALVA,AISGEO
3,0.2300,-7.266000,-35.892667,NDB - CAMPINA GRANDE NDB NOT AVBL BEYOND 60 NM,AISGEO
4,0.3400,3.860174,-51.800134,NDB - OIAPOQUE NDB NOT AVBL BEYOND 70 NM,AISGEO
5,0.2950,-19.658900,-43.896933,NDB - Lagoa Santa NDB NOT AVBL BEYOND 50 NM,AISGEO
6,0.2350,-19.765500,-47.959167,NDB - UBERABA NDB NOT AVBL BEYOND 50 NM,AISGEO
7,0.3200,-7.140500,-34.952500,NDB - JOÃO PESSOA,AISGEO
8,0.2750,-22.787975,-45.215286,NDB - GUARATINGUETÁ NDB NOT AVBL BEYOND 50 NM,AISGEO
9,0.3300,-21.142667,-47.776000,NDB - RIBEIRÃO NDB NOT AVBL BEYOND 50 NM,AISGEO


In [9]:
#| eval: false
get_geodf(LINK_DME, COLS_DME)

,Frequência,Latitude,Longitude,Entidade,Fonte
0,1001.0,-15.861500,-47.895000,DME - BRASÍLIA 40X,AISGEO
1,991.0,-8.722403,-63.900969,DME - PORTO VELHO 30X,AISGEO
2,1019.0,-22.812774,-42.095339,DME - ALDEIA 58X,AISGEO
3,1158.0,-5.197300,-37.364446,DME - MOSSORÓ 71X,AISGEO
4,991.0,-2.578333,-44.228333,DME - SÃO LUIZ 30X,AISGEO
...,...,...,...,...,...
160,1186.0,-14.907781,-40.918839,DME - VITÓRIA DA CONQUISTA 99X,AISGEO
161,1001.0,-3.772944,-38.541714,DME - FORTALEZA 40X,AISGEO
162,1176.0,-29.994028,-51.195874,DME - GUAIBA 89X,AISGEO
163,1166.0,-28.126178,-49.476596,DME - MORRO DA IGREJA 79X,AISGEO


In [10]:
#| export
def get_aisg() -> pd.DataFrame:  # DataFrame com todos os dados do GEOAISWEB
    """Lê e processa os dataframes individuais da API GEOAISWEB e retorna o conjunto concatenado"""
    df = pd.concat(
        get_geodf(link, cols)
        for link, cols in zip(
            [LINK_NDB, LINK_VOR, LINK_DME], [COLS_NDB, COLS_VOR, COLS_DME]
        )
    )
    return df #.astype("string").drop_duplicates(UNIQUE_COLS, ignore_index=True)

In [11]:
aisgeo = get_aisg()

In [12]:
aisgeo

,Frequência,Latitude,Longitude,Entidade,Fonte
0,0.1143,-32.340056,-54.223889,NDB - MELO,AISGEO
1,0.3000,-25.402616,-49.228917,NDB - BACACHERI NDB NOT AVBL BEYOND 50 NM,AISGEO
2,0.3400,-22.168824,-49.069677,NDB - AREALVA,AISGEO
3,0.2300,-7.266000,-35.892667,NDB - CAMPINA GRANDE NDB NOT AVBL BEYOND 60 NM,AISGEO
4,0.3400,3.860174,-51.800134,NDB - OIAPOQUE NDB NOT AVBL BEYOND 70 NM,AISGEO
...,...,...,...,...,...
160,1186.0000,-14.907781,-40.918839,DME - VITÓRIA DA CONQUISTA 99X,AISGEO
161,1001.0000,-3.772944,-38.541714,DME - FORTALEZA 40X,AISGEO
162,1176.0000,-29.994028,-51.195874,DME - GUAIBA 89X,AISGEO
163,1166.0000,-28.126178,-49.476596,DME - MORRO DA IGREJA 79X,AISGEO


In [14]:
aisgeo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 279 entries, 0 to 164
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   Frequência  279 non-null    float64 
 1   Latitude    279 non-null    float64 
 2   Longitude   279 non-null    float64 
 3   Entidade    279 non-null    string  
 4   Fonte       279 non-null    category
dtypes: category(1), float64(3), string(1)
memory usage: 11.3 KB


In [17]:
# AUTOGENERATED! DO NOT EDIT! File to edit: ../../nbs/02c_aisgeo.ipynb.

# %% auto 0
__all__ = [
    'LINK_VOR',
    'LINK_DME',
    'LINK_NDB',
    'COLS_VOR',
    'COLS_NDB',
    'COLS_DME',
    'UNIQUE_COLS',
    'convert_frequency',
    'get_geodf',
    'get_aisg',
]

# %% ../../nbs/02c_aisgeo.ipynb 2
import json
from typing import List
from urllib.request import urlopen
import polars as pl
from rfdatahub.constants import VOR_ILS_DME

# %% ../../nbs/02c_aisgeo.ipynb 5
LINK_VOR = 'https://geoaisweb.decea.mil.br/geoserver/ICA/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=ICA:vor&outputFormat=application%2Fjson'
LINK_DME = 'https://geoaisweb.decea.mil.br/geoserver/ICA/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=ICA:dme&outputFormat=application%2Fjson'
LINK_NDB = 'https://geoaisweb.decea.mil.br/geoserver/ICA/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=ICA:ndb&outputFormat=application%2Fjson'

COLS_VOR = (
    'properties.frequency', 'properties.frequnits',
    'properties.latitude', 'properties.longitude',
    'properties.tipo', 'properties.txtname', 'properties.txtrmk'
)

COLS_NDB = (
    'properties.valfreq', 'properties.uomfreq',
    'properties.geolat', 'properties.geolong',
    'properties.tipo', 'properties.txtname', 'properties.txtrmk'
)

COLS_DME = (
    'properties.valchannel', 'properties.codechanne',
    'properties.geolat', 'properties.geolong',
    'properties.tipo', 'properties.txtname', 'Channel'
)

UNIQUE_COLS = ['Frequência', 'Latitude', 'Longitude']

# %% ../../nbs/02c_aisgeo.ipynb 6
def convert_frequency(
    freq: pl.Expr,  # Frequência Central da emissão
    unit: pl.Expr,  # Unidade da Frequência
) -> pl.Expr:       # Frequência em MHz como expressão Polars
    """Converte a frequência para MHz usando expressões vetorizadas"""
    unit = unit.str.to_uppercase()
    return (
        pl.when(unit == "GHZ").then(freq * 1000)
        .when(unit == "MHZ").then(freq)
        .when(unit == "KHZ").then(freq / 1000)
        .when(unit == "HZ").then(freq / 1e6)
        .otherwise(pl.lit(None, pl.Float64))
    )

# %% ../../nbs/02c_aisgeo.ipynb 7
def _process_frequency(
    df: pl.DataFrame,  # Dataframe com os dados
    cols: List[str],   # Subconjunto de Colunas relevantes
) -> pl.DataFrame:
    if cols == COLS_DME:
        df_channels = pl.read_csv(
            VOR_ILS_DME,
            dtypes={"Channel": pl.Utf8, "DMEground": pl.Float64}
        )
        
        return (
            df.filter(pl.col(cols[0]).is_not_null())
            .with_columns(
                Channel=pl.col(cols[0]).cast(pl.Int64).cast(pl.Utf8) + pl.col(cols[1])
            )
            .join(
                df_channels.select("Channel", "DMEground"),
                on="Channel",
                how="left"
            )
            .rename({"DMEground": "Frequência"})
        )
    else:
        return df.with_columns(
            Frequência=convert_frequency(
                pl.col(cols[0]).cast(pl.Float64),
                pl.col(cols[1])
            )
        )

# %% ../../nbs/02c_aisgeo.ipynb 8
def _filter_df(df: pl.DataFrame, cols: List[str]) -> pl.DataFrame:
    return (
        df.with_columns(
            Entidade=(
                pl.col(cols[4]).fill_null("") + " - " +
                pl.col(cols[5]).fill_null("") + " " +
                pl.col(cols[6]).fill_null("")
            ).str.strip_chars(),
            Fonte=pl.lit("AISGEO").cast(pl.Categorical)
        )
        .select(
            "Frequência",
            pl.col(cols[2]).alias("Latitude").cast(pl.Float64),
            pl.col(cols[3]).alias("Longitude").cast(pl.Float64),
            "Entidade",
            "Fonte"
        )
    )

# %% ../../nbs/02c_aisgeo.ipynb 9
def get_geodf(
    link: str,   # Link para a requisição
    cols: List[str],  # Colunas relevantes
) -> pl.DataFrame:
    with urlopen(link) as response:
        if response.status != 200 or not response.headers.get('content-type', '').startswith('application/json'):
            raise ValueError(f'Resposta inválida: Status {response.status}')
        
        data = json.load(response)
    
    return (
        # pd.json_normalize(data['features'])
        pl.from_dicts(data['features'])
        # .select([pl.col(c) for c in cols])
        # .pipe(_process_frequency, cols=cols)
        # .pipe(_filter_df, cols=cols)
    )

# %% ../../nbs/02c_aisgeo.ipynb 13
def get_aisg() -> pl.DataFrame:
    """Combina dados de todas as fontes GEOAISWEB"""
    data_sources = [
        (LINK_NDB, COLS_NDB),
        (LINK_VOR, COLS_VOR),
        (LINK_DME, COLS_DME)
    ]
    
    return (
        pl.concat([get_geodf(link, cols) for link, cols in data_sources])
        .unique(subset=UNIQUE_COLS, maintain_order=True)
        .with_columns(Entidade=pl.col("Entidade").cast(pl.Utf8))
    )

In [18]:
df = get_geodf(LINK_VOR, COLS_VOR)
df

type,id,geometry,geometry_name,properties
str,str,struct[2],str,struct[59]
"""Feature""","""vor.10""","{""MultiPoint"",[[-54.5035, -25.5832]]}","""geom""","{10,10,""2023-10-05Z"",null,null,""FOZ"",-25.583203,-54.503514,""SAM"",""SB"",""FOZ"",-17.221107,""2023-10-05Z"",222.06,1.0,112.1,null,null,null,null,null,null,""MHZ"",null,""VOR"",null,""WGE"",1.0,""M"",5.072,null,""EGM_96"",null,""0000-0000"",null,57,null,null,null,null,""MAG"",null,null,""M"",""M"",""M"",""NO"",1.0,""YES"",null,null,null,null,""ALL"",""2025-01-23Z"",""2025-02-20Z"",""25°34'59.53"" S"",""054°30'12.65"" W"",""VOR""}"
"""Feature""","""vor.11""","{""MultiPoint"",[[-46.5282, -23.891]]}","""geom""","{11,11,""2023-10-05Z"",null,null,""RDE"",-23.890979,-46.528199,null,null,""REDE"",-21.822368,""2023-10-05Z"",790.04,null,116.7,null,null,null,null,null,null,""MHZ"",null,""VOR"",null,""WGE"",null,null,null,null,""EGM_96"",null,""0000-0000"",null,1,null,null,null,null,""MAG"",null,null,null,""M"",null,""NO"",null,""YES"",null,null,null,null,""ENROUTE"",""2025-01-23Z"",""2025-02-20Z"",""23°53'27.52"" S"",""046°31'41.52"" W"",""VOR""}"
"""Feature""","""vor.12""","{""MultiPoint"",[[-54.222, -32.3422]]}","""geom""","{12,12,""2012-01-01Z"",null,null,""MLO"",-32.342167,-54.222,null,null,""MELO"",-12.695329,""2012-01-01Z"",null,null,114.3,null,null,null,null,null,null,""MHZ"",null,""VOR"",null,""WGE"",null,null,null,null,""EGM_96"",null,""0000-0000"",null,90,null,null,null,null,""MAG"",null,null,null,null,null,null,null,null,null,null,null,null,""TERMINAL"",""2025-01-23Z"",""2025-02-20Z"",""32°20'31.80"" S"",""054°13'19.20"" W"",""VOR""}"
"""Feature""","""vor.13""","{""MultiPoint"",[[-38.3214, -12.9064]]}","""geom""","{13,13,""2023-10-05Z"",null,null,""SVD"",-12.906418,-38.321364,null,null,""SALVADOR"",-23.172328,""2023-10-05Z"",37.19,null,116.5,null,null,null,null,null,null,""MHZ"",null,""VOR"",null,""WGE"",null,null,null,null,""EGM_96"",null,""0000-0000"",null,116,null,null,null,null,""MAG"",null,null,null,""M"",null,""NO"",null,""YES"",null,null,null,null,""ALL"",""2025-01-23Z"",""2025-02-20Z"",""12°54'23.11"" S"",""038°19'16.91"" W"",""VOR""}"
"""Feature""","""vor.14""","{""MultiPoint"",[[-41.769, -22.3444]]}","""geom""","{14,15,""2024-11-28Z"",null,null,""MCA"",-22.344447,-41.769,null,null,""MACAÉ"",-23.494295,""2023-10-05Z"",2.13,null,115.9,null,null,null,null,null,null,""MHZ"",null,""VOR"",null,""WGE"",null,null,null,null,""EGM_96"",null,""0000-0000"",null,129,null,null,null,null,""MAG"",null,null,null,""M"",null,""NO"",null,""YES"",null,null,null,null,""ALL"",""2025-01-23Z"",""2025-02-20Z"",""22°20'40.01"" S"",""041°46'08.40"" W"",""VOR""}"
…,…,…,…,…
"""Feature""","""vor.69""","{""MultiPoint"",[[-61.0288, 4.693]]}","""geom""","{69,72,""2023-10-05Z"",null,null,""LDP"",4.693,-61.028833,null,null,""LA DIVINA PASTORA"",-15.803228,""2023-10-05Z"",98.0,1.0,117.7,null,null,null,null,null,null,""MHZ"",null,""DVOR"",null,""WGE"",0.5,""M"",null,null,""EGM_96"",null,""0000-0000"",null,90,null,null,null,null,""MAG"",null,null,""M"",""M"",null,""NO"",0.23,""YES"",null,null,null,null,""ALL"",""2025-01-23Z"",""2025-02-20Z"",""04°41'34.80"" N"",""061°01'43.80"" W"",""VOR""}"
"""Feature""","""vor.70""","{""MultiPoint"",[[-64.9383, -14.799]]}","""geom""","{70,73,""2012-01-01Z"",null,null,""TRI"",-14.799,-64.938333,null,null,""TRINIDAD-BL"",-9.956742,""2012-01-01Z"",null,null,115.9,null,null,null,null,null,null,""MHZ"",null,""VOR"",null,""WGE"",null,null,null,null,""EGM_96"",null,""0000-0000"",""For ADDN INFO, check Bolivia's AIP."",90,null,null,null,null,""MAG"",""Para INFO ADDN, consultar a AIP da Bolívia."",null,null,null,null,null,null,null,null,null,null,null,""ENROUTE"",""2025-01-23Z"",""2025-02-20Z"",""14°47'56.40"" S"",""064°56'18.00"" W"",""VOR""}"
"""Feature""","""vor.71""","{""MultiPoint"",[[-47.7703, -21.1428]]}","""geom""","{71,75,""2023-05-18Z"",null,null,""RPR"",-21.142778,-47.770278,null,null,""RIBEIRAO PRETO"",-22.0,""2023-01-01Z"",null,null,115.6,null,null,null,null

In [19]:
df.struct.unnest()

AttributeError: 'DataFrame' object has no attribute 'struct'